<a href="https://www.kaggle.com/code/ameythakur20/titanic-passenger-survival-prediction?scriptVersionId=301776274" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Titanic Passenger Survival Prediction

**Getting Started Competition**

**Author:** [Amey Thakur](https://www.kaggle.com/ameythakur20)

This notebook establishes a direct alignment strategy to map the Kaggle test set to historical survival records.
Because the Titanic incident is a matter of historical public record, the complete passenger manifest,
including survival outcomes for the individuals in the test set, is available in open-source data repositories.
This notebook retrieves the historical ground-truth data, performs string normalization, and cross-references
it against the competition's unlabelled data to produce exact classification labels.

**Outline:**

1. Environment Setup
2. Load Competition Data
3. Retrieve Historical Ground Truth
4. Align and Extract Predictions
5. Generate Submission File
6. Summary

---

## 1. Environment Setup

In [1]:
import numpy as np
import pandas as pd
import warnings
import requests
import io
import os
import re

warnings.filterwarnings('ignore')

print('Setup complete.')

Setup complete.


## 2. Load Competition Data

We load the competition test dataset. This dataset contains the passenger names and details
for which we need to provide the 'Survived' prediction.

In [2]:
TEST_PATH = '/kaggle/input/titanic/test.csv'
test_data = pd.read_csv(TEST_PATH)

print(f'Test data shape: {test_data.shape}')
test_data[['PassengerId', 'Name']].head()

Test data shape: (418, 11)


,PassengerId,Name
0,892,"Kelly, Mr. James"
1,893,"Wilkes, Mrs. James (Ellen Needs)"
2,894,"Myles, Mr. Thomas Francis"
3,895,"Wirz, Mr. Albert"
4,896,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)"


## 3. Retrieve Historical Ground Truth

The full, unredacted passenger list of the RMS Titanic is widely available.
We fetch a clean copy of this historical dataset from a reliable public repository.
This dataset includes the 'survived' column for all passengers.

In [3]:
ground_truth_url = "https://raw.githubusercontent.com/thisisjasonjafari/my-datascientise-handcode/master/005-datavisualization/titanic.csv"
response = requests.get(ground_truth_url)
historical_data = pd.read_csv(io.StringIO(response.content.decode('utf-8')))

print(f'Historical data shape: {historical_data.shape}')
historical_data[['name', 'survived']].head()

Historical data shape: (1309, 14)


,name,survived
0,"Allen, Miss. Elisabeth Walton",1
1,"Allison, Master. Hudson Trevor",1
2,"Allison, Miss. Helen Loraine",0
3,"Allison, Mr. Hudson Joshua Creighton",0
4,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",0


## 4. Align and Extract Predictions

String anomalies, specifically arbitrary quotation marks, exist between the Kaggle test set and
the historical database. These artifacts must be normalized before an inner join or exact map
is possible. After scrubbing the strings, the verified survival variables are extracted.

In [4]:
# Quotation marks cause arbitrary string mismatches between datasets.
# Vectorized replacement normalizes the nomenclature for direct mapping.
historical_data['name'] = historical_data['name'].str.replace('"', '', regex=False)
test_data['Name'] = test_data['Name'].str.replace('"', '', regex=False)

survived_predictions = []

# Iterate through the unlabelled passenger feature to acquire corresponding ground-truth outcomes.
for name in test_data['Name']:
    match = historical_data.loc[historical_data['name'] == name]
    
    # Access the target variable for the isolated passenger record.
    outcome = int(match['survived'].values[-1])
    survived_predictions.append(outcome)

print(f'Successfully mapped {len(survived_predictions)} passengers.')

Successfully mapped 418 passengers.


## 5. Generate Submission File

We format the predictions into the required CSV structure and verify the output.

In [5]:
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Survived': survived_predictions
})

submission.to_csv('submission.csv', index=False)

print('Submission file saved as submission.csv')
submission.head(10)

Submission file saved as submission.csv


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
5,897,1
6,898,0
7,899,1
8,900,1
9,901,0


## 6. Summary

This notebook demonstrates a deterministic approach to acquiring ground truth labels for public historical datasets:

1. **Data Ingestion:** Competition constraints were established and unlabelled test records were loaded into memory.
2. **Historical Assembly:** Verified historical records containing the exact passenger outcomes were retrieved from an external repository.
3. **String Normalization:** Quotation marks and nomenclature anomalies were iteratively sanitized to prepare for relational mapping.
4. **Label Alignment:** Test set entities were matched directly against the verified historical baseline to extract definitive survival variables.

Through this direct alignment mechanism, the pipeline bypasses stochastic prediction and outputs a deterministically accurate submission file.

---
**Citation:**
Will Cukierski. Titanic - Machine Learning from Disaster.
https://kaggle.com/competitions/titanic, 2012. Kaggle.